# Ad-level Threshold Tuning Notebook
- Load Ad-level table
- Explore distribution of key metrics
- Visualize natural clusters
- Define and tune thresholds for Local v National
- Keep Addressable as a separate concept

Key Features used:
- Covereage Score
- Entropy (Normalized)
- Significant DMAs
- DMA Mix Ratios


In [0]:
!pip install --upgrade matplotlib
%restart_python


In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

In [0]:
df = spark.table("dev.mohit_gangwani.ad_labeling_final_features_unbinned").toPandas()

In [0]:
df.fillna(0, inplace=True)

In [0]:
df.info()

In [0]:
df.rename(
    columns={
        "top5_dma_mix_ratio": "mix_ratio",
        "significant_dma_count_05": "sig_dma_count"
    },
    inplace=True,
)

In [0]:
df.describe()

In [0]:
metrics = [
    "coverage_score",
    "entropy_norm",
    "sig_dma_count",
    "mix_ratio"
]
for col in metrics:
    df.loc[:, col] = df[col].astype(float)

In [0]:
ndf = df[df["total_impressions"] >= 1000].copy()
ndf.reset_index(drop=True, inplace=True)

In [0]:
df.head(20)

In [0]:
metrics = [
    "coverage_score",
    "entropy_norm",
    "sig_dma_count",
    "mix_ratio"
]
for col in metrics:
    plt.figure()
    plt.hist(df[col], bins=50)
    # plt.yscale("log")
    plt.title(f"Distribution of {col.replace('_', ' ').title()}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.show()

In [0]:
metrics = [
    "coverage_score",
    "entropy_norm",
    "sig_dma_count",
    "mix_ratio"
]
for col in metrics:
    plt.figure()
    plt.hist(ndf[col], bins=50)
    # plt.yscale("log")
    plt.title(f"Distribution of {col.replace('_', ' ').title()}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.show()

In [0]:
qs = [0.05, 0.1, 0.25, 0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.90, 0.95]
quantiles = df[metrics].quantile(qs)
display(quantiles)

In [0]:
# qs = [0.05, 0.1, 0.25, 0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.90, 0.95]
qs = [i/20 for i in range(1, 20)]
quantiles = ndf[metrics].quantile(qs)
display(quantiles)

In [0]:
def plot_a_vs_b(df_plot, a, b, a_name, b_name):
    plt.figure(figsize=(8, 7))

    # for t in df_plot['ad_type'].unique():
    #     subset = df_plot[df_plot['ad_type'] == t]
    plt.scatter(
        df_plot[a],
        df_plot[b],
        alpha=0.25,
        s=2,
    )

    plt.ylabel(b_name)
    plt.xlabel(a_name)
    plt.title(f'{a_name} vs {b_name}')
    plt.legend()
    plt.show()

In [0]:
plot_dict = {
    "coverage_score": "Coverage Score",
    "entropy_norm": "Entropy Norm",
    "mix_ratio": "Top 5 DMA Mix Ratio",
    "sig_dma_count": "Significant DMAs",
}
for a, v1 in plot_dict.items():
    for b, v2 in plot_dict.items():
        if a != b:
            plot_a_vs_b(
                df_plot=ndf,
                a=a,
                b=b,
                a_name=v1,
                b_name=v2,
            )

In [0]:
def plot_kde(df: pd.DataFrame, x: str, y: str, x_name: str, y_name: str):
    sns.kdeplot(
        data=df,
        x=x,
        y=y,
        levels=100,
        thresh=0.02,
        fill=False,
        alpha=0.4,
    )

    plt.title(f"KDE Plot - {x_name} vs {y_name}")
    plt.xlabel(x_name)
    plt.ylabel(y_name)
    plt.show()

In [0]:
plot_dict = {
    "coverage_score": "Coverage Score",
    "entropy_norm": "Entropy Norm",
    "mix_ratio": "Top 5 DMA Mix Ratio",
    "sig_dma_count": "Significant DMAs",
}
done = []
for x, v1 in plot_dict.items():
    for y, v2 in plot_dict.items():
        if x != y and sorted([x, y]) not in done:
            try:
                done.append(sorted([x, y]))
                plot_kde(df=ndf, x=x, y=y, x_name=v1, y_name=v2)
            except ValueError:
                print(f'failed {x} {y}')

In [0]:
def classify_ads_v1(row):
    cov = row["coverage_score"]
    ent = row["entropy_norm"]
    mix = row["mix_ratio"]
    sig = row["sig_dma_count"]

    # ----- HARDCODED COVERAGE OVERRIDES -----
    # If extremely low coverage, always local
    if cov < 0.05 and mix >= 0.95:
        return "Local"
    elif cov < 0.05 and ent <= 0.50:
        return "Local"
    elif ent < 0.50 and mix >= 0.95:
        return "Local"
    elif cov <= 0.025:
        return "Local"
    elif cov <= 0.005 and ent == 0.000 and mix >= 0.90:
        return "Local"

    # If sufficiently high coverage, always national
    if cov >= 0.70 and mix <= 0.25:
        return "National"
    elif cov >= 0.70 and ent >= 0.85:
        return "National"
    elif ent >= 0.85 and mix <= 0.25:
        return "National"
    elif cov >= 0.97:
        return "National"
    elif cov >= 0.25 and ent >= 0.875 and mix <= 0.125:
        return "National"

    return 'Mixed'

In [0]:
ndf["ad_type"] = ndf.apply(classify_ads_v1, axis=1)

nat = ndf[ndf["ad_type"] == "National"]
loc = ndf[ndf["ad_type"] == "Local"]
mix = ndf[ndf["ad_type"] == "Mixed"]
len(nat), len(loc), len(mix)

In [0]:
def print_seed_quantiles(nat_seed, loc_seed, mix_seed=None):
    cols = [
        "coverage_score",
        "entropy_norm",
        "sig_dma_count",
        'mix_ratio'
    ]
    nat_seed_float = nat_seed[cols].astype(float)
    loc_seed_float = loc_seed[cols].astype(float)
    mix_seed_float = mix_seed[cols].astype(float)

    print("=== National seed quantiles ===")
    quant = [i/20 for i in range(1, 20)]
    # display(nat_seed_float.quantile([0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95]))
    display(nat_seed_float.quantile(quant))

    print("\n=== Local seed quantiles ===")
    display(loc_seed_float.quantile(quant))

    if mix_seed is not None:
        mix_seed_float = mix_seed[cols].astype(float)
        print("\n=== Mixed seed quantiles ===")
        display(mix_seed_float.quantile(quant))

print_seed_quantiles(nat, loc, mix)

In [0]:
ndf[ndf.ad_id.isin(['AE16546-2025-46-01049', 'AE16546-2025-02-02070', 'AE16546-2025-18-06039', 'AE16546-2025-18-05561', 'AE16546-2025-31-02225', 'AE16546-2025-15-04183', 'AE16546-2019-45-02522', 'AE16546-2025-36-05079', 'AE16546-2025-35-03273'])].display()

In [0]:
ndf[ndf.ad_id.isin(['AE16546-2025-44-05535', 'AE16546-2025-32-04566', 'AE16546-2025-33-03327', 'AE16546-2025-50-02565', 'AE16546-2025-06-04375', 'AE16546-2025-40-06605', 'AE16546-2025-49-03587', 'AE16546-2025-27-01472', 'AE16546-2025-50-00743', 'AE16546-2025-47-17764', 'AE16546-2025-27-01881', 'AE16546-2025-48-10057'])].display()

In [0]:
def summarize_quantiles(df, label):
    print(f"=== {label} quantiles ===")
    display(
        df[metrics].astype(float).quantile(
            [0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95]
        )
    )

In [0]:
def add_localness_score(df: pd.DataFrame) -> pd.DataFrame:
    # x = df.copy()

    # Rank-based normalization is robust to seasonality & saturation.
    # Higher score => more local.
    df.loc[:, "cov_rank"] = df["coverage_score"].rank(pct=True)  # high -> national
    df.loc[:, "ent_rank"] = df["entropy_norm"].rank(pct=True)  # high -> national
    df.loc[:, "mix_rank"] = df["mix_ratio"].rank(pct=True)  # high -> local

    # Convert to "local direction"
    cov_local = 1.0 - df["cov_rank"]
    ent_local = 1.0 - df["ent_rank"]
    mix_local = df["mix_rank"]

    # Weighted average (entropy important, mix very important, coverage important)
    df.loc[:, "localness_score"] = 0.40 * mix_local + 0.25 * ent_local + 0.35 * cov_local
    return df

In [0]:
ndf = add_localness_score(ndf)
df = add_localness_score(df)

In [0]:
ndf[ndf.ad_type == 'National'].localness_score.hist(bins=20)

In [0]:
summarize_quantiles(ndf, "National")

In [0]:
cols = [
    "coverage_score",
    "entropy_norm",
    "mix_ratio",
    'localness_score'
]
nat_seed_float = ndf[ndf.ad_type == 'National'][cols].astype(float)
loc_seed_float = ndf[ndf.ad_type == 'Local'][cols].astype(float)

print("=== National seed quantiles ===")
quant = [i/20 for i in range(1, 20)]
# display(nat_seed_float.quantile([0.05, 0.10, 0.25, 0.5, 0.75, 0.90, 0.95]))
display(nat_seed_float.quantile(quant))

print("\n=== Local seed quantiles ===")
display(loc_seed_float.quantile(quant))

In [0]:
nat = ndf[ndf.ad_type == "National"].copy()
loc = ndf[ndf.ad_type == "Local"].copy()
cutoffs = {
    "national_local_score": nat.localness_score.quantile(0.95),
    "local_local_score": loc.localness_score.quantile(0.05)
}

In [0]:
def classify_binary_v1(row):
    cov = row["coverage_score"]
    ent = row["entropy_norm"]
    mix = row["mix_ratio"]
    local = row["localness_score"]

    # ----- HARDCODED COVERAGE OVERRIDES -----
    # If extremely low coverage, always local
    if cov <= 0.05 and mix >= 0.95:
        return "Local"
    elif cov <= 0.05 and ent <= 0.50:
        return "Local"
    elif ent <= 0.50 and mix >= 0.95:
        return "Local"
    elif cov <= 0.025:
        return "Local"
    elif cov <= 0.005 and ent == 0.000 and mix >= 0.90:
        return "Local"

    # If sufficiently high coverage, always national
    if cov >= 0.70 and ent >= 0.85:
        return "National"
    elif cov >= 0.70 and mix <= 0.25:
        return "National"
    elif ent >= 0.80 and mix <= 0.30:
        return "National"
    elif cov >= 0.95:
        return "National"
    elif cov >= 0.25 and ent >= 0.875 and mix <= 0.125:
        return "National"

    if ent >= 0.975:
        return "National"
    elif ent <= 0.025:
        return "Local"

    if local >= cutoffs['local_local_score']:
        return "Local"
    elif local <= cutoffs['national_local_score']:
        return "National"

    return 'Mixed'

In [0]:
ndf["ad_type_bin"] = ndf.apply(classify_binary_v1, axis=1)
ndf.groupby('ad_type_bin').ad_id.count().sort_values(ascending=False).reset_index().display()

In [0]:
nat = ndf[ndf.ad_type_bin == "National"].copy()
loc = ndf[ndf.ad_type_bin == "Local"].copy()
cutoffs = {
    "national_local_score": nat.localness_score.quantile(0.975),
    "local_local_score": loc.localness_score.quantile(0.005),
    "national_coverage": nat.coverage_score.quantile(0.10),
    "local_coverage": loc.coverage_score.quantile(0.90),
    "national_entropy": nat.entropy_norm.quantile(0.10),
    "local_entropy": loc.entropy_norm.quantile(0.90),
    "national_mix_ratio": nat.mix_ratio.quantile(0.90),
    "local_mix_ratio": loc.mix_ratio.quantile(0.10),
}

In [0]:
cutoffs

In [0]:
def classify_binary_final(row):
    if row["ad_type_bin"] == "National":
        return "National"
    elif row["ad_type_bin"] == "Local":
        return "Local"

    cov = row["coverage_score"]
    ent = row["entropy_norm"]
    mix = row["mix_ratio"]
    local = row["localness_score"]

    if local >= cutoffs["local_local_score"]:
        return "Local"
    elif local <= cutoffs["national_local_score"]:
        return "National"

    # ----- NATIONAL CONDITIONS -----
    nat_conds = [
        cov >= cutoffs["national_coverage"],
        ent >= cutoffs["national_entropy"],
        mix <= cutoffs["national_mix_ratio"],
    ]
    nat_score = sum(nat_conds)

    # ----- LOCAL CONDITIONS -----
    loc_conds = [
        cov <= cutoffs["local_coverage"],
        ent <= cutoffs["local_entropy"],
        mix >= cutoffs["local_mix_ratio"],
    ]
    loc_score = sum(loc_conds)

    # ----- DECISION (NO MIXED) -----
    # Strong votes
    if nat_score >= 2:
        return "National"
    elif loc_score >= 2:
        return "Local"
    # -- Set Numbers --
    # First pass at 
    elif local > 0.6:
        return "Local"
    elif local < 0.4:
        return "National"
    
    # -- From Cutoffs --
    elif cov <= cutoffs["local_coverage"]:
        return "Local"
    elif cov >= cutoffs["national_coverage"]:
        return "National"
    elif ent <= cutoffs["local_entropy"]:
        return "Local"
    elif ent >= cutoffs["national_entropy"]:
        return "National"
    elif mix >= cutoffs["local_mix_ratio"]:
        return "Local"
    elif mix <= cutoffs["national_mix_ratio"]:
        return "National"
    
    # -- Final Binary --
    elif local > 0.5:
        return "Local"
    return "National"

In [0]:
ndf["ad_type_final"] = ndf.apply(classify_binary_final, axis=1)
ndf.groupby('ad_type_final').ad_id.count().sort_values(ascending=False).reset_index().display()

In [0]:
df["ad_type"] = df.apply(classify_ads_v1, axis=1)
df = add_localness_score(df)
df["ad_type_bin"] = df.apply(classify_binary_v1, axis=1)
df["ad_type_final"] = df.apply(classify_binary_final, axis=1)

In [0]:
df.groupby('ad_type_final').ad_id.count().sort_values(ascending=False).reset_index().display()

In [0]:
# ndf["ad_type_bin"] = ndf.apply(classify_binary_v1, axis=1)

# nat = ndf[ndf["ad_type_bin"] == "National"]
# loc = ndf[ndf["ad_type_bin"] == "Local"]
# mix = ndf[ndf["ad_type_bin"] == "Mix"]
# len(nat), len(loc), len(mix)

In [0]:
# df["ad_type_bin"] = df.apply(classify_binary_v1, axis=1)

# nat = df[df["ad_type_bin"] == "National"]
# loc = df[df["ad_type_bin"] == "Local"]
# mix = df[df["ad_type_bin"] == "Mix"]
# len(nat), len(loc), len(mix)

In [0]:
nat = ndf[ndf["ad_type_final"] == "National"]
loc = ndf[ndf["ad_type_final"] == "Local"]

In [0]:
nat = df[df["ad_type_final"] == "National"]
loc = df[df["ad_type_final"] == "Local"]

In [0]:
summarize_quantiles(nat, "National")
summarize_quantiles(loc, "Local")

In [0]:
def plot_a_vs_b(df_plot, a, b, a_name, b_name):
    plt.figure(figsize=(8, 7))

    for t in df_plot['ad_type_final'].unique():
        subset = df_plot[df_plot['ad_type_final'] == t]
        plt.scatter(
            subset[a],
            subset[b],
            alpha=0.25,
            label=t,
            s=2,
        )

    plt.ylabel(b_name)
    plt.xlabel(a_name)
    plt.title(f'{a_name} vs {b_name}')
    plt.legend()
    plt.show()

In [0]:

plot_dict = {
    "coverage_score": [0.020, 0.057, "Coverage Score"],
    "entropy_norm": [0.794, 0.609, "Entropy Norm"],
    "mix_ratio": [2.96, 3.314, "DMA Mix Ratio"],
    "localness_score": [5.528, 2.155, "Localness Score"],
}
for a, v1 in plot_dict.items():
    for b, v2 in plot_dict.items():
        if a != b:
            plot_a_vs_b(
                df_plot=ndf,
                a=a,
                b=b,
                a_name=v1[2],
                b_name=v2[2],
            )

In [0]:
ndf[(ndf.ad_type_final == 'Local') & (ndf.sig_dma_count >= 15)].display()


In [0]:
def plot_kde(df: pd.DataFrame, x: str, y: str, x_name: str, y_name: str):
    sns.kdeplot(
        data=df,
        x=x,
        y=y,
        hue='ad_type_final',
        levels=100,
        thresh=0.02,
        fill=False,
        alpha=0.4,
    )

    plt.title(f"KDE Plot - {x_name} vs {y_name}")
    plt.xlabel(x_name)
    plt.ylabel(y_name)
    plt.show()

In [0]:
plot_dict = {
    "coverage_score": "Coverage Score",
    "entropy_norm": "Entropy Norm",
    "mix_ratio": "DMA Mix Ratio",
    "localness_score": "Localness Score",
}
done = []
for x, v1 in plot_dict.items():
    for y, v2 in plot_dict.items():
        if x != y and sorted([x, y]) not in done:
            try:
                done.append(sorted([x, y]))
                plot_kde(df=ndf, x=x, y=y, x_name=v1, y_name=v2)
            except ValueError:
                print(f'failed {x} {y}' )

In [0]:
def plot_boxplot_by_label(df, col, label_col="ad_type_final"):
    data = [
        df[df[label_col] == "Local"][col].dropna(),
        df[df[label_col] == "National"][col].dropna()
        ]
    labels = ["Local", "National"]

    plt.figure(figsize=(6, 5))
    plt.boxplot(data, labels=labels, showfliers=False)
    plt.title(f"{col} distribution by label")
    plt.ylabel(col)
    plt.xlabel("ad_type")
    plt.grid(True, axis="y")
    plt.show()

In [0]:
def plot_hist_by_label(df, col, label_col="ad_type_final", bins=50):
    plt.figure(figsize=(7, 5))
    for label in ["Local", "National"]:
        subset = df[df[label_col] == label][col].dropna()
        plt.hist(subset, bins=bins, alpha=0.5, label=label)

    plt.xlabel(col)
    plt.ylabel("Count")
    plt.title(f"{col} histogram by label")
    plt.legend()
    plt.grid(True)
    plt.show()

In [0]:
for col in ["coverage_score", "entropy_norm", "mix_ratio"]:
    plot_boxplot_by_label(ndf, col)

In [0]:
for col in ["coverage_score", "entropy_norm", "mix_ratio"]:
    plot_hist_by_label(ndf, col)

In [0]:
ndf.columns

In [0]:
final_df = df[
    ["ad_id",
    "total_impressions",
    "total_opportunities",
    "coverage_score",
    "entropy_norm",
    "mix_ratio",
    "sig_dma_count",
    "localness_score",
    "ad_type_final"]
].copy()

In [0]:
ndf[ndf.ad_id == 'AE16546-2023-39-01586'].display()

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_national_local_set_12_16_25;

In [0]:
spark_df = spark.createDataFrame(final_df)
spark_df.write.option("mergeSchema", "true").mode("overwrite").saveAsTable("dev.mohit_gangwani.ad_labeling_national_local_set_12_16_25_new")


In [0]:
def load_and_transform(table_name: str):
    df = spark.table(table_name).toPandas()

    df.fillna(0, inplace=True)
    df.rename(
        columns={
            "top5_dma_mix_ratio": "mix_ratio",
            "significant_dma_count_05": "sig_dma_count",
        },
        inplace=True,
    )

    metrics = ["coverage_score", "entropy_norm", "mix_ratio"]

    for col in metrics:
        df.loc[:, col] = df[col].astype(float)

    ndf = df[df["total_impressions"] >= 1000].copy()
    ndf.reset_index(drop=True, inplace=True)

    return df, ndf


def classify_ads_v1(row):
    cov = row["coverage_score"]
    ent = row["entropy_norm"]
    mix = row["mix_ratio"]

    # ----- HARDCODED COVERAGE OVERRIDES -----
    # If extremely low coverage, always local
    if (
        (cov < 0.05 and mix >= 0.95)
        or (cov < 0.05 and ent <= 0.50)
        or (ent < 0.50 and mix >= 0.95)
        or (cov <= 0.025)
        or (cov <= 0.005 and ent == 0.000 and mix >= 0.90)
    ):
        return "Local"

    # If sufficiently high coverage, always national
    elif (
        (cov >= 0.70 and mix <= 0.25)
        or (cov >= 0.70 and ent >= 0.85)
        or (ent >= 0.85 and mix <= 0.25)
        or (cov >= 0.97)
        or (cov >= 0.25 and ent >= 0.875 and mix <= 0.125)
    ):
        return "National"

    return "Mixed"


def add_localness_score(df: pd.DataFrame) -> pd.DataFrame:
    # Rank-based normalization is robust to seasonality & saturation
    # Higher score = more local
    df.loc[:, "cov_rank"] = df["coverage_score"].rank(pct=True)
    df.loc[:, "ent_rank"] = df["entropy_norm"].rank(pct=True)
    df.loc[:, "mix_rank"] = df["mix_ratio"].rank(pct=True)

    # Convert to "local direction"
    cov_local = 1.0 - df["cov_rank"]
    ent_local = 1.0 - df["ent_rank"]
    mix_local = df["mix_rank"]

    # Weighted average (entropy important, mix very important, coverage important)
    df.loc[:, "localness_score"] = (
        0.40 * mix_local + 0.30 * ent_local + 0.30 * cov_local
    )
    return df


def cutoffs_first_iter(nat: pd.DataFrame, loc: pd.DataFrame) -> dict:
    cutoffs = {
        "national_local_score": nat.localness_score.quantile(0.95),
        "local_local_score": loc.localness_score.quantile(0.05),
    }

    return cutoffs


def classify_binary_v1(row):
    cov = row["coverage_score"]
    ent = row["entropy_norm"]
    mix = row["mix_ratio"]
    local = row["localness_score"]

    # ----- HARDCODED COVERAGE OVERRIDES -----
    # If extremely low coverage, always local
    if (
        (cov <= 0.05 and mix >= 0.95)
        or (cov <= 0.05 and ent <= 0.50)
        or (ent <= 0.50 and mix >= 0.95)
        or (cov <= 0.025)
        or (cov <= 0.005 and ent == 0.000 and mix >= 0.90)
    ):
        return "Local"

    # If sufficiently high coverage, always national
    if (
        (cov >= 0.70 and ent >= 0.85)
        or (cov >= 0.70 and mix <= 0.25)
        or (ent >= 0.80 and mix <= 0.30)
        or (cov >= 0.95)
        or (cov >= 0.25 and ent >= 0.875 and mix <= 0.125)
    ):
        return "National"

    if ent >= 0.975:
        return "National"
    elif ent <= 0.025:
        return "Local"

    if local >= cutoffs["local_local_score"]:
        return "Local"
    elif local <= cutoffs["national_local_score"]:
        return "National"

    return "Mixed"


def cutoffs_second_iter(nat: pd.DataFrame, loc: pd.DataFrame) -> dict:
    cutoffs = {
        "national_local_score": nat.localness_score.quantile(0.975),
        "local_local_score": loc.localness_score.quantile(0.005),
        "national_coverage": nat.coverage_score.quantile(0.10),
        "local_coverage": loc.coverage_score.quantile(0.90),
        "national_entropy": nat.entropy_norm.quantile(0.10),
        "local_entropy": loc.entropy_norm.quantile(0.90),
        "national_mix_ratio": nat.mix_ratio.quantile(0.90),
        "local_mix_ratio": loc.mix_ratio.quantile(0.10),
    }

    return cutoffs


def classify_binary_final(row):
    if row["ad_type_bin"] == "National":
        return "National"
    elif row["ad_type_bin"] == "Local":
        return "Local"

    cov = row["coverage_score"]
    ent = row["entropy_norm"]
    mix = row["mix_ratio"]
    local = row["localness_score"]

    nat_cov_co = cutoffs["national_coverage"]
    loc_cov_co = cutoffs["local_coverage"]
    nat_ent_co = cutoffs["national_entropy"]
    loc_ent_co = cutoffs["local_entropy"]
    loc_mix_co = cutoffs["local_mix_ratio"]
    nat_mix_co = cutoffs["national_mix_ratio"]

    if local >= cutoffs["local_local_score"]:
        return "Local"
    elif local <= cutoffs["national_local_score"]:
        return "National"

    # ----- NATIONAL CONDITIONS -----
    nat_conds = [cov >= nat_cov_co, ent >= nat_ent_co, mix <= nat_mix_co]
    nat_score = sum(nat_conds)

    # ----- LOCAL CONDITIONS -----
    loc_conds = [cov <= loc_cov_co, ent <= loc_ent_co, mix >= loc_mix_co]
    loc_score = sum(loc_conds)

    # ----- DECISION (NO MIXED) -----
    # Strong votes
    if nat_score >= 2:
        return "National"
    elif loc_score >= 2:
        return "Local"
    # -- Set Numbers --
    elif local > 0.6:
        return "Local"
    elif local < 0.4:
        return "National"

    # -- From Cutoffs --
    elif cov <= loc_cov_co or ent <= loc_ent_co or mix >= loc_mix_co:
        return "Local"
    elif cov >= nat_cov_co or ent >= nat_ent_co or mix <= nat_mix_co:
        return "National"

    # -- Final Binary --
    elif local > 0.5:
        return "Local"
    return "National"

In [0]:
table_name = 'dev.mohit_gangwani.ad_labeling_final_features_unbinned'
df, ndf = load_and_transform(table_name)

ndf = add_localness_score(ndf)
df = add_localness_score(df)

ndf["ad_type"] = ndf.apply(classify_ads_v1, axis=1)
df["ad_type"] = df.apply(classify_ads_v1, axis=1)

nat = ndf[ndf.ad_type == "National"].copy()
loc = ndf[ndf.ad_type == "Local"].copy()

cutoffs = cutoffs_first_iter(nat=nat, loc=loc)

ndf["ad_type_bin"] = ndf.apply(classify_binary_v1, axis=1)
df["ad_type_bin"] = df.apply(classify_binary_v1, axis=1)

nat = ndf[ndf.ad_type_bin == "National"].copy()
loc = ndf[ndf.ad_type_bin == "Local"].copy()

cutoffs = cutoffs_second_iter(nat=nat, loc=loc)

ndf["ad_type_final"] = ndf.apply(classify_binary_final, axis=1)
df["ad_type_final"] = df.apply(classify_binary_final, axis=1)

final_df = df[
    [
        "ad_id",
        "total_impressions",
        "total_opportunities",
        "coverage_score",
        "entropy_norm",
        "mix_ratio",
        "sig_dma_count",
        "localness_score",
        "ad_type_final",
    ]
].copy()

spark_df = spark.createDataFrame(final_df)

In [0]:
%sql
DROP TABLE IF EXISTS dev.mohit_gangwani.ad_labeling_national_local_set_12_16_25_new;

In [0]:
spark_df.write.option("mergeSchema", "true").mode("overwrite").saveAsTable("dev.mohit_gangwani.ad_labeling_national_local_set_12_16_25_new")